## Getting Started: install required packages

In [1]:
!pip install -q bitsandbytes transformers peft accelerate datasets sentencepiece scikit-learn torch trl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.1/60.1 MB 12.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 564.6/564.6 kB 25.8 MB/s eta 0:00:00


### Mount Google Drive

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


### Login to HugginFace

In [3]:
!huggingface-cli login

⚠️  Warning: 'huggingface-cli login' is deprecated. Use 'hf auth login' instead.

    _|    _|  _|    _|    _|_|_|    _|_|_|  _|_|_|  _|      _|    _|_|_|      _|_|_|_|    _|_|      _|_|_|  _|_|_|_|
    _|    _|  _|    _|  _|        _|          _|    _|_|    _|  _|            _|        _|    _|  _|        _|
    _|_|_|_|  _|    _|  _|  _|_|  _|  _|_|    _|    _|  _|  _|  _|  _|_|      _|_|_|    _|_|_|_|  _|        _|_|_|
    _|    _|  _|    _|  _|    _|  _|    _|    _|    _|    _|_|  _|    _|      _|        _|    _|  _|        _|
    _|    _|    _|_|      _|_|_|    _|_|_|  _|_|_|  _|      _|    _|_|_|      _|        _|    _|    _|_|_|  _|_|_|_|

    To log in, `huggingface_hub` requires a token generated from https://huggingface.co/settings/tokens .
Enter your token (input will not be visible): 
Add token as git credential? (Y/n) n
Token is valid (permission: fineGrained).
The token `ldm-datasets` has been saved to /root/.cache/huggingface/stored_tokens
Your token has been saved to /ro

### Import Libraries

In [4]:
import pandas as pd
import json
import csv
import random
import os
import torch
import numpy as np
from collections import deque
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, accuracy_score
from datasets import Dataset
from functools import partial
import bitsandbytes as bnb

from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    set_seed,
    Trainer,
    TrainingArguments,
    BitsAndBytesConfig,
    DataCollatorForLanguageModeling
)
from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training,
    AutoPeftModelForCausalLM
)

# Set seed for reproducibility
SEED = 42
set_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)

print("Dataset downloaded. Processing format...")

Dataset downloaded. Processing format...


In [5]:
def preprocess_kg_file(input_file, output_file):
    with open(input_file, 'r') as f:
        lines = f.readlines()

    with open(output_file, 'w') as f:
        f.write(f"{len(lines)}\n")
        for line in lines:
            parts = line.strip().split('\t')
            if len(parts) == 3:
                # Convert to format: node1 node2 relation_id
                f.write(f"{parts[0]} {parts[1]} {parts[2]}\n")

preprocess_kg_file('/content/drive/MyDrive/Experiemental_Data/NeSyKGLLM/LC_original/train2id.txt', '/content/drive/MyDrive/Experiemental_Data/NeSyKGLLM/LC_original/train2id_processed.txt')


In [8]:
relations = set()
with open('/content/drive/MyDrive/Experiemental_Data/NeSyKGLLM/LC_original/train2id_processed.txt', 'r') as f:
    n = int(f.readline())
    for line in f:
        parts = line.strip().split()
        if len(parts) == 3:
            relations.add(parts[2])

with open('/content/drive/MyDrive/Experiemental_Data/NeSyKGLLM/LC_original/relation2id.txt', 'w') as f:
    f.write(f"{len(relations)}\n")
    for i, rel in enumerate(sorted(relations)):
        f.write(f"relation_{rel}\t{rel}\n")

print(f"Processed {n} triples with {len(relations)} unique relations")


Processed 11768 triples with 9 unique relations


In [9]:
def load_knowledge_graph(file_path):
    """Load knowledge graph into memory"""
    graph = {}
    nodes = set()

    with open(file_path, 'r') as f:
        num_lines = int(f.readline())
        for line in f:
            parts = line.strip().split()
            if len(parts) == 3:
                node1, node2, relation = parts
                nodes.add(node1)

                if node1 not in graph:
                    graph[node1] = {}
                graph[node1][node2] = int(relation)

    return graph, list(nodes)

def load_relation_mapping(file_path):
    """Load relation ID to name mapping"""
    relation2id = {}
    with open(file_path, 'r') as f:
        num_relations = int(f.readline())
        for line in f:
            relation, relation_id = line.strip().split('\t')
            relation2id[int(relation_id)] = relation
    return relation2id

def generate_training_data(graph, node_list, relation2id,
                          total_samples=1000, max_path_length=10,
                          include_reasoning=True):
    """Generate training data for link prediction"""

    data = []
    unique_paths = set()
    pos_count = 0
    neg_count = 0

    while len(data) < total_samples:
        # Random path length
        path_length = random.randint(2, max_path_length)

        # Start from random node
        first_node = random.choice(node_list)
        visited = {first_node}
        path_text = ""
        reasoning_text = ""
        previous_node = first_node

        # Build path
        for step in range(path_length - 1):
            if previous_node not in graph or not graph[previous_node]:
                # Dead end - add disconnected node
                node = random.choice(node_list)
                while node in visited:
                    node = random.choice(node_list)
                path_text += f'node_{previous_node} not connected with node_{node}. '
                if include_reasoning:
                    reasoning_text += f'node_{previous_node} not connected with node_{node} means there is no relationship. '
                visited.add(node)
                previous_node = node
            else:
                # Follow edge
                next_node = random.choice(list(graph[previous_node].keys()))
                while next_node in visited and len(visited) < len(node_list):
                    next_node = random.choice(list(graph[previous_node].keys()))

                relation = graph[previous_node][next_node]
                rel_name = relation2id.get(relation, f"relation_{relation}")
                path_text += f'node_{previous_node} has {rel_name} with node_{next_node}. '
                if include_reasoning:
                    reasoning_text += f'node_{previous_node} has {rel_name} with node_{next_node}. '
                visited.add(next_node)
                previous_node = next_node

        last_node = previous_node

        # Skip if we've seen this path
        if path_text in unique_paths:
            continue
        unique_paths.add(path_text)

        # Create question
        question = f'Is node_{first_node} connected with node_{last_node}?'

        # Determine answer
        is_connected = (first_node in graph and last_node in graph[first_node]) or \
                      (last_node in graph and first_node in graph[last_node])

        # Balance positive and negative examples
        if is_connected and pos_count >= total_samples // 2:
            continue
        if not is_connected and neg_count >= total_samples // 2:
            continue

        if is_connected:
            if include_reasoning:
                answer = reasoning_text + 'The answer is yes.'
            else:
                answer = 'The answer is yes.'
            pos_count += 1
        else:
            if include_reasoning:
                answer = reasoning_text + 'The answer is no.'
            else:
                answer = 'The answer is no.'
            neg_count += 1

        # Format prompt
        if include_reasoning:
            prompt = f"###Instruction:\nAnswer the following yes/no question by reasoning step-by-step.\n\n###Input:\n{path_text}{question}\n\n###Response:\n{answer}"
        else:
            prompt = f"###Input:\n{path_text}{question}\n\n###Response:\n{answer}"

        data.append({
            'Prompt': prompt,
            'input_text': path_text + question,
            'output_text': answer
        })

    return pd.DataFrame(data)

In [10]:
print("Loading knowledge graph...")
graph, node_list = load_knowledge_graph('/content/drive/MyDrive/Experiemental_Data/NeSyKGLLM/LC_original/train2id_processed.txt')
relation2id = load_relation_mapping('/content/drive/MyDrive/Experiemental_Data/NeSyKGLLM/LC_original/relation2id.txt')

print(f"Graph loaded: {len(graph)} nodes, {len(node_list)} total nodes")

# Generate training data (with reasoning - CoT style)
print("Generating training data...")
train_df = generate_training_data(
    graph, node_list, relation2id,
    total_samples=2000,  # Increase for better results
    max_path_length=8,
    include_reasoning=True
)

# Generate test data
print("Generating test data...")
test_df = generate_training_data(
    graph, node_list, relation2id,
    total_samples=500,
    max_path_length=8,
    include_reasoning=True
)

# Save datasets
train_df.to_csv('/content/drive/MyDrive/Experiemental_Data/NeSyKGLLM/LC_original/train_data.csv', index=False)
test_df.to_csv('/content/drive/MyDrive/Experiemental_Data/NeSyKGLLM/LC_original/test_data.csv', index=False)

print(f"Training samples: {len(train_df)}")
print(f"Test samples: {len(test_df)}")
print("\nSample training example:")
print(train_df.iloc[0]['Prompt'][:500] + "...")


Loading knowledge graph...
Graph loaded: 1242 nodes, 1242 total nodes
Generating training data...
Generating test data...
Training samples: 2000
Test samples: 500

Sample training example:
###Instruction:
Answer the following yes/no question by reasoning step-by-step.

###Input:
node_1124 has relation_7 with node_1341. node_1341 not connected with node_1077. node_1077 has relation_1 with node_1242. node_1242 not connected with node_978. node_978 has relation_0 with node_1320. node_1320 not connected with node_367. Is node_1124 connected with node_367?

###Response:
node_1124 has relation_7 with node_1341. node_1341 not connected with node_1077 means there is no relationship. node_...


In [11]:
def create_bnb_config():
    """Create 4-bit quantization config"""
    return BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16,
    )

def load_model(model_name, bnb_config):
    """Load model and tokenizer"""
    n_gpus = torch.cuda.device_count()
    max_memory = f'{40960}MB'

    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        quantization_config=bnb_config,
        device_map="auto",
        max_memory={i: max_memory for i in range(n_gpus)},
    )
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    tokenizer.pad_token = tokenizer.eos_token

    return model, tokenizer

def create_peft_config(modules):
    """Create LoRA config"""
    return LoraConfig(
        r=16,
        lora_alpha=64,
        target_modules=modules,
        lora_dropout=0.1,
        bias="none",
        task_type="CAUSAL_LM",
    )

def find_all_linear_names(model):
    """Find all linear layers for LoRA"""
    cls = bnb.nn.Linear4bit
    lora_module_names = set()
    for name, module in model.named_modules():
        if isinstance(module, cls):
            names = name.split('.')
            lora_module_names.add(names[0] if len(names) == 1 else names[-1])

    if 'lm_head' in lora_module_names:
        lora_module_names.remove('lm_head')
    return list(lora_module_names)

def preprocess_batch(batch, tokenizer, max_length=512):
    """Tokenize batch"""
    return tokenizer(
        batch["Prompt"],
        truncation=True,
        max_length=max_length,
        padding='max_length'
    )


def evaluate_model(model, tokenizer, test_df, max_samples=200, device='cuda'):
    """Evaluate model on test set"""
    model.eval()
    y_true = []
    y_pred = []

    print(f"Evaluating on {min(len(test_df), max_samples)} samples...")

    for idx, row in test_df.head(max_samples).iterrows():
        input_text = "###Input:\n" + row['input_text']
        expected_has_yes = 'yes' in row['output_text'].lower()

        # Generate prediction
        inputs = tokenizer(input_text, return_tensors="pt").to(device)
        with torch.no_grad():
            outputs = model.generate(
                input_ids=inputs["input_ids"],
                attention_mask=inputs["attention_mask"],
                max_new_tokens=150,
                pad_token_id=tokenizer.eos_token_id,
                do_sample=False
            )

        model_answer = tokenizer.decode(outputs[0], skip_special_tokens=True)
        model_has_yes = 'yes' in model_answer.lower()

        y_true.append(expected_has_yes)
        y_pred.append(model_has_yes)

        if (idx + 1) % 50 == 0:
            print(f"Processed {idx + 1} samples...")

    # Calculate metrics
    accuracy = accuracy_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred, pos_label=True)

    return {
        'accuracy': accuracy,
        'f1_score': f1,
        'y_true': y_true,
        'y_pred': y_pred
    }


def fine_tune_model(model, tokenizer, train_dataset, output_dir, num_steps=500):
    """Fine-tune model with LoRA"""

    # Prepare model
    model.gradient_checkpointing_enable()
    model = prepare_model_for_kbit_training(model)

    # Get LoRA modules
    modules = find_all_linear_names(model)
    peft_config = create_peft_config(modules)
    model = get_peft_model(model, peft_config)

    # Print trainable parameters
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    all_params = sum(p.numel() for p in model.parameters())
    print(f"Trainable params: {trainable_params:,} ({100 * trainable_params / all_params:.2f}%)")

    # Training arguments
    training_args = TrainingArguments(
        per_device_train_batch_size=1,
        gradient_accumulation_steps=4,
        warmup_steps=10,
        max_steps=num_steps,
        learning_rate=2e-4,
        fp16=True,
        logging_steps=50,
        output_dir=output_dir,
        optim="paged_adamw_8bit",
        save_strategy="steps",
        save_steps=100,
    )

    # Create trainer
    trainer = Trainer(
        model=model,
        train_dataset=train_dataset,
        args=training_args,
        data_collator=DataCollatorForLanguageModeling(tokenizer, mlm=False)
    )

    model.config.use_cache = False

    # Train
    print("Starting training...")
    trainer.train()

    # Save model
    print("Saving model...")
    os.makedirs(output_dir, exist_ok=True)
    trainer.model.save_pretrained(output_dir)

    return model


In [19]:
MODELS = {
    'LLaMA-3-8B': 'meta-llama/Llama-3.2-1B-Instruct',
    'EuroLLM-9B': 'utter-project/EuroLLM-9B'
}

# Choose which model to use (change this to test different models)
MODEL_TO_USE = 'LLaMA-3-8B'  # or 'EuroLLM-9B'

print(f"\n{'='*60}")
print(f"COMPARING: {MODEL_TO_USE}")
print(f"{'='*60}\n")

model_name = MODELS[MODEL_TO_USE]
bnb_config = create_bnb_config()


COMPARING: LLaMA-3-8B



## Step 1: Evaluate LLMs without fine-tuning

In [20]:

print("\n" + "="*60)
print("STEP 1: Evaluating BASE MODEL (no fine-tuning)")
print("="*60 + "\n")

base_model, base_tokenizer = load_model(model_name, bnb_config)
base_results = evaluate_model(base_model, base_tokenizer, test_df, max_samples=200)

print(f"\n📊 BASE MODEL RESULTS:")
print(f"   Accuracy: {base_results['accuracy']:.3f}")
print(f"   F1 Score: {base_results['f1_score']:.3f}")



STEP 1: Evaluating BASE MODEL (no fine-tuning)



config.json:   0%|          | 0.00/877 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.47G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/54.5k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Evaluating on 200 samples...
Processed 50 samples...
Processed 100 samples...
Processed 150 samples...
Processed 200 samples...

📊 BASE MODEL RESULTS:
   Accuracy: 0.215
   F1 Score: 0.256


In [21]:
del base_model
torch.cuda.empty_cache()

## Step 2: Fine-tune Model

In [22]:
print("\n" + "="*60)
print("STEP 2: FINE-TUNING MODEL")
print("="*60 + "\n")

# Load fresh model for training
ft_model, ft_tokenizer = load_model(model_name, bnb_config)

# Prepare dataset
train_dataset = Dataset.from_pandas(train_df)
train_dataset = train_dataset.map(
    lambda batch: preprocess_batch(batch, ft_tokenizer),
    batched=True,
)

# Fine-tune (increase num_steps for better results, e.g., 500-1000)
output_dir = f"finetuned_{MODEL_TO_USE.replace('-', '_')}"
ft_model = fine_tune_model(
    ft_model,
    ft_tokenizer,
    train_dataset,
    output_dir,
    num_steps=100  # Increase this for better results
)


STEP 2: FINE-TUNING MODEL



Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Trainable params: 11,272,192 (1.48%)
Starting training...


/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)


<IPython.core.display.Javascript object>

...

 ··········


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: yashrajsinh-chudasama (yashrajsinh-chudasama-tib) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:929: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss
50,0.492500
100,0.240300


Saving model...


In [23]:
print("\n" + "="*60)
print("STEP 3: Evaluating FINE-TUNED MODEL")
print("="*60 + "\n")

ft_results = evaluate_model(ft_model, ft_tokenizer, test_df, max_samples=200)

print(f"\n📊 FINE-TUNED MODEL RESULTS:")
print(f"   Accuracy: {ft_results['accuracy']:.3f}")
print(f"   F1 Score: {ft_results['f1_score']:.3f}")



STEP 3: Evaluating FINE-TUNED MODEL

Evaluating on 200 samples...
Processed 50 samples...
Processed 100 samples...
Processed 150 samples...
Processed 200 samples...

📊 FINE-TUNED MODEL RESULTS:
   Accuracy: 0.375
   F1 Score: 0.277


## Comparison of LLMs with/without fine-tuning

In [24]:
print("\n" + "="*60)
print("FINAL COMPARISON RESULTS")
print("="*60 + "\n")

comparison_df = pd.DataFrame({
    'Model': ['Base Model', 'Fine-tuned Model'],
    'Accuracy': [base_results['accuracy'], ft_results['accuracy']],
    'F1 Score': [base_results['f1_score'], ft_results['f1_score']],
    'Improvement': ['—', f"+{(ft_results['accuracy'] - base_results['accuracy']) * 100:.1f}%"]
})

print(comparison_df.to_string(index=False))

# Calculate improvement
acc_improvement = (ft_results['accuracy'] - base_results['accuracy']) * 100
f1_improvement = (ft_results['f1_score'] - base_results['f1_score']) * 100

print(f"\n📈 IMPROVEMENTS:")
print(f"   Accuracy: +{acc_improvement:.1f} percentage points")
print(f"   F1 Score: +{f1_improvement:.1f} percentage points")

# Save results
results_summary = {
    'model': MODEL_TO_USE,
    'base_accuracy': float(base_results['accuracy']),
    'base_f1': float(base_results['f1_score']),
    'finetuned_accuracy': float(ft_results['accuracy']),
    'finetuned_f1': float(ft_results['f1_score']),
    'accuracy_improvement': float(acc_improvement),
    'f1_improvement': float(f1_improvement)
}

with open('comparison_results.json', 'w') as f:
    json.dump(results_summary, f, indent=2)

print("\n✅ Results saved to comparison_results.json")
print(f"✅ Fine-tuned model saved to {output_dir}/")


FINAL COMPARISON RESULTS

           Model  Accuracy  F1 Score Improvement
      Base Model     0.215  0.255924           —
Fine-tuned Model     0.375  0.277457      +16.0%

📈 IMPROVEMENTS:
   Accuracy: +16.0 percentage points
   F1 Score: +2.2 percentage points

✅ Results saved to comparison_results.json
✅ Fine-tuned model saved to finetuned_LLaMA_3_8B/


In [26]:
!zip -r wandb.zip wandb
from google.colab import files
files.download("wandb.zip")

  adding: wandb/ (stored 0%)
  adding: wandb/debug-internal.log (deflated 68%)
  adding: wandb/run-20251006_132501-1yfb8n9h/ (stored 0%)
  adding: wandb/run-20251006_132501-1yfb8n9h/tmp/ (stored 0%)
  adding: wandb/run-20251006_132501-1yfb8n9h/tmp/code/ (stored 0%)
  adding: wandb/run-20251006_132501-1yfb8n9h/run-1yfb8n9h.wandb (deflated 83%)
  adding: wandb/run-20251006_132501-1yfb8n9h/logs/ (stored 0%)
  adding: wandb/run-20251006_132501-1yfb8n9h/logs/debug-internal.log (deflated 68%)
  adding: wandb/run-20251006_132501-1yfb8n9h/logs/debug-core.log (deflated 56%)
  adding: wandb/run-20251006_132501-1yfb8n9h/logs/debug.log (deflated 69%)
  adding: wandb/run-20251006_132501-1yfb8n9h/files/ (stored 0%)
  adding: wandb/run-20251006_132501-1yfb8n9h/files/output.log (deflated 68%)
  adding: wandb/run-20251006_132501-1yfb8n9h/files/wandb-metadata.json (deflated 42%)
  adding: wandb/run-20251006_132501-1yfb8n9h/files/requirements.txt (deflated 56%)
  adding: wandb/latest-run/ (stored 0%)
  a

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Now integrating VANILLA approach